# 01 · Base comparison, HRL-H profile and extended metrics
Table and figure numbers refer to manuscript draft v5, in which all tables are in Section 5 (there is no appendix). All figures are saved to `../figures/` at 600 dpi.

| Figure file | Illustrates (draft v5) | Style |
|---|---|---|
| `Table09_hrlh_radar.png` | Table 9: HRL-H in the four environments | radar, 4 lines |
| `Table10_base_matrix.png` | Table 10: 8 methods × 7 metrics × 4 environments | annotated performance matrix |
| `Table10_base_radar.png` | Table 10 (overall profile per environment) | radar per environment |
| `Table11-12_extended_matrix.png` | Tables 11 and 12: reward, safety, information age, latency, blocked moves | annotated performance matrix |

Matrix colour = rank of the method within the column of an environment (green best, red worst, grey = not directional or all equal); the printed number is the mean over 10 seeds.

In [ ]:
import sys
sys.path.insert(0, "..")          # common.py lives in the package root
from common import *
%matplotlib inline

## Table 9 · HRL-H profile across the four environments
Scores are relative to the best value of each metric over **all** methods and environments, so the differences between environments stay visible.

In [ ]:
allrec = [rec(BASE, e, m) for e in ENVS for m in METH]
ref = dict(cov=max(r["coverage"] for r in allrec), stp=min(r["steps"] for r in allrec), en=min(r["energy"] for r in allrec),
           ds=min(r["dist"] for r in allrec), so=max(r["soc"] for r in allrec))
sc = profile_scores([rec(BASE, e, "hrlh") for e in ENVS], ref)
env_col = ["#d62728", "#1f77b4", "#2ca02c", "#9467bd"]
fig = plt.figure(figsize=(5.2, 3.9))
ax = fig.add_subplot(111, projection="polar")
ang = radar_axes_setup(ax, fs=6.6)
for i, e in enumerate(ENVS):
    r = np.r_[sc[i], sc[i][0]]
    ax.plot(ang, r, color=env_col[i], lw=1.6, marker="o", ms=3, label=ENVN[e])
    ax.fill(ang, r, color=env_col[i], alpha=0.06)
ax.legend(loc="center left", bbox_to_anchor=(1.12, 0.5), fontsize=7, frameon=False)
fig.subplots_adjust(left=0.07, right=0.72, top=0.90, bottom=0.10)
save(fig, "Table09_hrlh_radar")

## Table 10 · annotated performance matrix (7 metrics × 8 methods, one panel per environment)

In [ ]:
MET = [  # key, column label, higher_is_better, formatter
    ("coverage", "Coverage\n(%) ↑", True, lambda v: f"{v:.0f}"),
    ("stranded", "Stranded\n(%) ↓", False, lambda v: f"{v:.0f}"),
    ("steps", "Episode\nlength ↓", False, lambda v: f"{v:.0f}"),
    ("energy_wh_per_task", "Energy\n(Wh/task) ↓", False, lambda v: f"{v:.2f}"),
    ("dist_per_task_m", "Distance\n(m/task) ↓", False, lambda v: f"{v/1000:.1f}k" if v >= 1000 else f"{v:.0f}"),
    ("soc", "Final\nSOC ↑", True, lambda v: f"{v:.2f}"),
    ("contention_pct", "Contention\n(%) ↓", False, lambda v: f"{v:.0f}"),
]

def matrix_grid(src, keys_rows, row_labels, hl_row, MET, name, figsize, envs=ENVS, bar_label=("worst rank", "best rank")):
    fig, axs = plt.subplots(2, 2, figsize=figsize)
    for k, e in enumerate(envs):
        ax = axs[k // 2, k % 2]
        V = np.array([[bm(e, m, key, src) for key, *_ in MET] for m in keys_rows], float)
        S = np.column_stack([rank_score(V[:, j], MET[j][2]) for j in range(len(MET))])
        annotated_matrix(ax, V, S, [f for *_, f in MET], row_labels, [l for _, l, *_ in MET], hl_row=hl_row)
        ax.set_title(f"({'abcd'[k]}) {ENVN[e]}", fontsize=8, pad=22)
    fig.subplots_adjust(left=0.085, right=0.995, top=0.90, bottom=0.075, wspace=0.20, hspace=0.42)
    cax = fig.add_axes([0.32, 0.03, 0.36, 0.014])
    cb = fig.colorbar(plt.cm.ScalarMappable(cmap="RdYlGn", norm=plt.Normalize(0, 1)), cax=cax, orientation="horizontal")
    cb.set_ticks([0, 1]); cb.set_ticklabels(list(bar_label)); cb.ax.tick_params(labelsize=6.4, length=0); cb.outline.set_visible(False)
    save(fig, name)

matrix_grid(BASE, METH, [LAB[m] for m in METH], hl_row=len(METH) - 1, MET=MET, name="Table10_base_matrix", figsize=(6.9, 5.4))

## Table 10 · radar per environment
Each axis is scored relative to the best method in that environment (1 = best): coverage/best, 1 − stranded, best steps/steps, best energy/energy, best distance/distance, SOC/best, 1 − contention.

In [ ]:
def radar_grid(series, names, cols, styles, ref_fn, name, title_fn, figsize=(6.9, 6.4), legend_cols=8):
    fig = plt.figure(figsize=figsize)
    for k, e in enumerate(ENVS):
        ax = fig.add_subplot(2, 2, k + 1, projection="polar")
        recs = series(e)
        sc = profile_scores(recs, ref_fn(e) if ref_fn else None)
        ang = radar_axes_setup(ax)
        for i, nm in enumerate(names):
            r = np.r_[sc[i], sc[i][0]]
            ax.plot(ang, r, color=cols[i], lw=styles[i][0], ls=styles[i][1], label=nm, zorder=styles[i][2])
            if styles[i][2] >= 10: ax.fill(ang, r, color=cols[i], alpha=0.12)
        ax.set_title(title_fn(k, e), fontsize=8, pad=20)
    h, l = ax.get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=legend_cols, fontsize=6.6, frameon=False, bbox_to_anchor=(0.5, 0.0), columnspacing=1.0, handlelength=1.6)
    fig.subplots_adjust(left=0.06, right=0.94, top=0.92, bottom=0.08, wspace=0.30, hspace=0.42)
    save(fig, name)

sty = {m: ((2.0, "-", 10) if m == "hrlh" else (1.5, "-", 6) if m == "dmpc" else (0.8, "-" if m in ("greedy", "cbba") else "--", 4)) for m in METH}
radar_grid(lambda e: [rec(BASE, e, m) for m in METH], [LAB[m] for m in METH], [COL[m] for m in METH], [sty[m] for m in METH],
           None, "Table10_base_radar", lambda k, e: f"({'abcd'[k]}) {ENVN[e]}")

## Tables 11 + 12 · extended metrics (reward, safety, information age, latency, blocked moves)
Information age is not a performance direction, so it is drawn in neutral grey; columns whose values are all equal (e.g. blocked moves without NFZs) are grey as well.

In [ ]:
MET2 = [
    ("reward", "Team\nreward ↑", True, lambda v: f"{v:.0f}"),
    ("min_soc", "Minimum\nSOC ↑", True, lambda v: f"{v:.2f}"),
    ("below_floor_pct", "Below floor\n(%) ↓", False, lambda v: f"{v:.1f}"),
    ("info_age", "Info age\n(steps)", None, lambda v: f"{v:.1f}"),
    ("latency_mean", "Latency\nmean ↓", False, lambda v: f"{v:.1f}"),
    ("latency_median", "Latency\nmedian ↓", False, lambda v: f"{v:.1f}"),
    ("blocked_moves", "Blocked\nmoves ↓", False, lambda v: f"{v:.0f}"),
]
fig, axs = plt.subplots(2, 2, figsize=(6.9, 5.4))
for k, e in enumerate(ENVS):
    ax = axs[k // 2, k % 2]
    V = np.array([[bm(e, m, key) for key, *_ in MET2] for m in METH], float)
    S = np.column_stack([np.full(len(METH), np.nan) if MET2[j][2] is None else rank_score(V[:, j], MET2[j][2]) for j in range(len(MET2))])
    annotated_matrix(ax, V, S, [f for *_, f in MET2], [LAB[m] for m in METH], [l for _, l, *_ in MET2], hl_row=7)
    ax.set_title(f"({'abcd'[k]}) {ENVN[e]}", fontsize=8, pad=22)
fig.subplots_adjust(left=0.085, right=0.995, top=0.90, bottom=0.075, wspace=0.20, hspace=0.42)
cax = fig.add_axes([0.32, 0.03, 0.36, 0.014])
cb = fig.colorbar(plt.cm.ScalarMappable(cmap="RdYlGn", norm=plt.Normalize(0, 1)), cax=cax, orientation="horizontal")
cb.set_ticks([0, 1]); cb.set_ticklabels(["worst rank", "best rank"]); cb.ax.tick_params(labelsize=6.4, length=0); cb.outline.set_visible(False)
save(fig, "Table11-12_extended_matrix")